# 🧠 Brain Tumor CNN - Final Working Version
## Complete Training Pipeline

In [1]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import numpy as np
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

print('✅ Libraries imported!')

✅ Libraries imported!


In [ ]:
import pickle
from PIL import Image

DATASET_PATH = r'D:\brain_tumor_mri\new_dataset'
IMAGES_PATH = os.path.join(DATASET_PATH, 'bt_images')
LABELS_FILE = os.path.join(DATASET_PATH, 'labels.pickle')

IMG_SIZE = (150, 150)

# Load labels
with open(LABELS_FILE, 'rb') as f:
    labels = pickle.load(f)
labels = np.array(labels)

# Load and resize images
image_files = sorted([f for f in os.listdir(IMAGES_PATH) if f.endswith(('.jpg', '.png', '.jpeg'))])
X = np.zeros((len(image_files), 150, 150, 3), dtype=np.uint8)

for idx, img_file in enumerate(image_files):
    img_path = os.path.join(IMAGES_PATH, img_file)
    try:
        img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
        X[idx] = np.array(img)
    except Exception as e:
        print(f"Skip {img_file}: {e}")

print(f'✅ X shape: {X.shape}')
print(f'✅ Labels: {len(labels)}')
print(f'✅ Unique labels: {np.unique(labels)}')

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3064,) + inhomogeneous part.

In [ ]:
# Preprocess
IMG_SIZE = (150, 150)
NUM_CLASSES = len(np.unique(labels))

# Resize
if X.shape[1:3] != IMG_SIZE:
    from PIL import Image
    X_resized = []
    for img in X:
        pil_img = Image.fromarray(img.astype('uint8')).resize(IMG_SIZE)
        X_resized.append(np.array(pil_img))
    X = np.array(X_resized)

# Convert to RGB if needed
if X.ndim == 3:
    X = np.stack([X, X, X], axis=-1)

# Normalize
X = X.astype('float32') / 255.0

# Encode labels
unique_labels = sorted(list(set(labels)))
label_to_int = {lbl: idx for idx, lbl in enumerate(unique_labels)}
y_int = np.array([label_to_int[lbl] for lbl in labels])
y_onehot = to_categorical(y_int, num_classes=NUM_CLASSES)

print(f'✅ Preprocessed! X: {X.shape}, y: {y_onehot.shape}')

In [ ]:
# Split data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_onehot, test_size=0.30, random_state=42, stratify=y_int
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
# Augmentation
train_datagen = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.1, horizontal_flip=True, fill_mode='nearest'
)

train_gen = train_datagen.flow(X_train, y_train, batch_size=32, seed=42)

print('✅ Augmentation ready!')

In [ ]:
# Build model
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3), padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy', metrics=['accuracy'])

print('✅ Model built!')

In [ ]:
# TRAINING - THIS IS THE KEY CELL
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_brain_tumor_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Class weights for imbalance
class_weights = {i: len(y_int) / (NUM_CLASSES * np.sum(y_int == i)) for i in range(NUM_CLASSES)}

history = model.fit(
    train_gen,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

print('✅ Training Complete!')

In [ ]:
# Evaluate
best_model = keras.models.load_model('best_brain_tumor_model.h5')
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)

print(f'Test Accuracy: {test_acc*100:.2f}%')
print(f'Test Loss: {test_loss:.4f}')

In [ ]:
# Save model
best_model.save(r'D:\brain_tumor_mri\model_final_best.h5')
print('✅ Model saved as model_final_best.h5!')

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report

y_pred = best_model.predict(X_test, verbose=0)
y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

print(classification_report(y_true_labels, y_pred_labels))

cm = confusion_matrix(y_true_labels, y_pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png')
plt.close()

print('✅ Confusion matrix saved!')

In [ ]:
# Training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()

plt.savefig('training_history.png')
plt.close()

print('✅ All done!')